# Mock B · Grade ceramic tiles

An inspection line records five measurements and three assessor scores for each tile. Derive three grades, handle two missing sensors and evaluate a prescribed classifier. The split sizes and grade thresholds are specific to this mock.

**Practice conditions** · 180 minutes · 30 practice points. Start with a fresh kernel. Run the supplied setup, then implement each task in its answer cell. Checks run after you fill the preceding cell. Grade the printed examples and plots yourself. Use array operations across records. Loops over batches, epochs, a few printed records and plot annotations are fine.

All data are generated locally. Use Python, NumPy, PyTorch and Matplotlib. The worked notebook is in `solutions/` for review after your attempt.

In [ ]:
import copy
import csv
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)
torch.manual_seed(41)

In [ ]:
# Supplied input. Two sensors use zero to indicate a missing measurement.
def tile_export():
    rng = np.random.default_rng(48)
    x = rng.normal(size=(1493, 5))
    signal = x[:, 0] - x[:, 2] + 0.8*x[:, 3] - 0.4*np.abs(x[:, 4])
    scores = np.clip(np.rint(5 + signal[:, None] + rng.normal(0, 1.4, (len(x), 3))), 1, 9)
    x[:, :2] += 5
    x[:, :2] = np.where(rng.random((len(x), 2)) < 0.14, 0, x[:, :2])
    data = np.column_stack((x, scores))
    text = StringIO()
    np.savetxt(text, data, delimiter=",", header="mass,gloss,grain,tint,warp,r1,r2,r3",
               comments="", fmt="%.4f")
    return text.getvalue()

content = tile_export()
print(content.splitlines()[:3])

## A · Read the inspection log · 1 point

Parse the comma-separated string into `raw` with dtype float32. Skip the header and retain all numeric fields, including the zeros.

In [ ]:
# Create raw.

In [ ]:
# Check your work
assert raw.shape == (1493, 8) and raw.dtype == np.float32 and np.isfinite(raw).all()
assert np.allclose(raw[1], np.fromstring(content.splitlines()[2], sep=","))
print("Check passed.")

## B · Encode sensor availability · up to 5 points

Choose one route. For 0 points, keep the original table. For 3, insert two validity flags at positions 2 and 3, handling sensors separately. For 5, compute both flags in one vectorized operation. A flag is 1 when that sensor is present and 0 when missing. Preserve original values, including zero sentinels. Store the result in `table`.

In [ ]:
# Create table.

In [ ]:
# Check your work
assert table.dtype == np.float32 and len(table) == len(raw)
if table.shape[1] == 8:
    assert np.array_equal(raw, table)
    print("Carry-forward route, 0 points for B.")
else:
    assert table.shape[1] == 10
    assert np.array_equal(table[:, 2:4], raw[:, :2] != 0)
    assert np.array_equal(table[:, :2], raw[:, :2]) and np.array_equal(table[:, 4:], raw[:, 2:])
print("Check passed.")

## C · Allocate fixed holdouts · 3 points

Use one permutation of complete rows. Reserve exactly 231 tiles for validation and 257 for test. Use the remainder for training. Store arrays as `train, val, test` and print their shapes.

In [ ]:
# Shuffle and split.

In [ ]:
# Check your work
assert (len(train), len(val), len(test)) == (1005, 231, 257)
together = np.concatenate((train, val, test))
assert not np.array_equal(together, table)
for actual, expected in zip(np.unique(together, axis=0, return_counts=True), np.unique(table, axis=0, return_counts=True)):
    assert np.array_equal(actual, expected)
print("Check passed.")

## D · Derive grades and build loaders · 6 points

Grade 0 if any assessor gives a score **less than 3**. Grade 2 if every score is **at least 4** and their mean is **at least 6**. Give remaining tiles grade 1. Implement `grades(scores)` for an N × 3 score matrix. Exclude all assessor scores from the inputs. Build `train_loader, val_loader, test_loader` with mixed batches of 32 and keep partial batches. Inputs are float32 (B, d), labels are int64 (B,). Print counts in class order 0, 1, 2 for each split.

In [ ]:
def grades(scores):
    raise NotImplementedError

# Build loaders and print class distributions.

In [ ]:
# Check your work
assert np.array_equal(grades(np.array([[2, 9, 9], [3, 8, 9], [4, 6, 8], [4, 5, 8]])), [0, 1, 2, 1])
for loader, part in zip((train_loader, val_loader, test_loader), (train, val, test)):
    batches = list(loader)
    x, y = (torch.cat(v) for v in zip(*batches))
    assert len(batches[0][0]) == 32 and any(labels.unique().numel() == 3 for _, labels in batches)
    assert x.shape == (len(part), table.shape[1]-3) and x.dtype == torch.float32
    assert y.shape == (len(part),) and y.dtype == torch.int64
    s = part[:, -3:]
    expected_y = np.where(s.min(1) <= 2, 0, np.where((s.min(1) >= 4) & (s.sum(1) >= 18), 2, 1))
    expected = np.column_stack((part[:, :-3], expected_y))
    actual = np.column_stack((x.numpy(), y.numpy()))
    for v, w in zip(np.unique(actual, axis=0, return_counts=True), np.unique(expected, axis=0, return_counts=True)):
        assert np.array_equal(v, w)
print("Check passed.")

## E · Define the MLP · 3 points

Use precisely two linear layers, with 20 hidden units, ReLU, then dropout with probability 0.10, followed by three output logits. Define `Net(n_inputs)` as an `nn.Module`, infer `n_inputs` from a training batch and instantiate `model`. Verify its output on that batch.

In [ ]:
class Net(nn.Module):
    # Implement __init__ and forward.
    pass

# Set n_inputs and instantiate model.

In [ ]:
# Check your work
xb, _ = next(iter(train_loader))
assert model(xb).shape == (len(xb), 3)
assert sum(p.numel() for p in model.parameters()) == (n_inputs+1)*20 + 21*3
assert any(isinstance(layer, nn.Dropout) and layer.p == 0.1 for layer in model.modules())
print("Check passed.")

## F · Train, restore and evaluate · 6 points

Train the model on `train_loader` with Adam and `nn.CrossEntropyLoss()`. Record one sample-weighted mean loss per epoch for each of training and validation. Use a maximum of 100 epochs and stop after eight consecutive epochs without a strictly lower validation loss. Save an independent snapshot on improvement and automatically restore the minimum-validation-loss weights, including when the epoch cap is reached.

Plot both loss histories together with a legend. In evaluation mode, gather aligned inputs, labels and logits for **all** test batches in a single pass. Report mean test loss and accuracy, then print five predicted/true label pairs. Keep the original input values for inspection. Use `y_test`, `x_test` and `logits_test` for your collected tensors, and `pred` for decisions with the same shape as `y_test`. Use `train_loss`, `val_loss`, `best_epoch` (zero-based), `test_loss` and `test_acc` for the other results. Finally build a 3 × 3 confusion matrix named `cm` with true rows and predicted columns. Plot it with axis labels, class ticks and the count in every cell.

In [ ]:
# Implement training, validation, restoration, plotting and test evaluation here.

In [ ]:
# Check your work
assert len(train_loss) == len(val_loss) and 1 <= len(val_loss) <= 100
assert np.isfinite(train_loss).all() and np.isfinite(val_loss).all()
assert best_epoch == np.argmin(val_loss)
assert len(val_loss) == 100 or len(val_loss) - 1 - best_epoch == 8
assert len(y_test) == len(test_loader.dataset) == len(x_test)
assert pred.shape == y_test.shape and not model.training
with torch.no_grad():
    assert torch.allclose(model(x_test), logits_test, atol=1e-5)
    ref_criterion = nn.CrossEntropyLoss()
    ref_val = sum(ref_criterion(model(x), y).item() * len(y) for x, y in val_loader)
assert np.isclose(ref_val / len(val_loader.dataset), min(val_loss), atol=1e-5)
assert np.isclose(test_loss, ref_criterion(logits_test, y_test).item(), atol=1e-5)
assert torch.equal(pred, logits_test.argmax(dim=1))
assert np.isclose(test_acc, (pred == y_test).float().mean().item())
assert np.array_equal(np.asarray(cm), np.histogram2d(y_test, pred, bins=[np.arange(4)-0.5]*2)[0])
print("Check passed.")

## G · Compute the specified macro accuracy · 3 + 3 points

Write `macro_ovr(y, pred, k)`. For each class c, regard c as positive and all other classes as negative. Calculate (TP + TN) / N for c, then average over all k classes. Return the per-class values in class order and their mean. Use NumPy or PyTorch. Earn 3 points for a correct solution with no loop over samples, plus 3 for no loop over classes either. Print both results on the test set. Hand-check the tiny fixture before running the check.

In [ ]:
def macro_ovr(y, pred, k):
    raise NotImplementedError

# Report the per-class and macro test results.

In [ ]:
# Check your work
fixture_y = [0, 0, 1, 1, 2, 2]
fixture_pred = [0, 1, 1, 2, 2, 1]
by_class, avg = macro_ovr(fixture_y, fixture_pred, 3)
assert np.allclose(by_class, [5/6, 3/6, 4/6]) and np.isclose(avg, 2/3)
by_class, avg = macro_ovr(y_test, pred, 3)
counts = np.asarray(cm)
ref = 1 - (counts.sum(0) + counts.sum(1) - 2*counts.diagonal()) / len(y_test)
assert np.allclose(by_class, ref) and np.isclose(avg, ref.mean())
print("Check passed.")